In [1]:
import pandas as pd
import sqlite3

listing_master = pd.read_csv("../data/London/enriched/listing_master.csv")

listing_master.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,booked_days,occupancy_rate,estimated_revenue,neighbourhood_median_price,neighbourhood_listing_count,neighbourhood_avg_rating,host_tenure_years,listing_age_years,review_frequency,price_per_bedroom
0,13913,https://www.airbnb.com/rooms/13913,20250914034649,2025-09-16,city scrape,Holiday London DB Room Let-on going,My bright double bedroom with a large window h...,Finsbury Park is a friendly melting pot commun...,https://a0.muscache.com/pictures/miso/Hosting-...,54730,...,34,9.315068,2380.0,141.0,5036,4.695029,16.60,15.846680,3.470758,70.0
1,15400,https://www.airbnb.com/rooms/15400,20250914034649,2025-09-16,city scrape,Bright Chelsea Apartment. Chelsea!,Lots of windows and light. St Luke's Gardens ...,It is Chelsea.,https://a0.muscache.com/pictures/428392/462d26...,60302,...,166,45.479452,24734.0,225.0,6401,4.674686,16.55,16.503765,5.877447,149.0
2,17402,https://www.airbnb.com/rooms/17402,20250914034649,2025-09-16,city scrape,Very Central Modern 3-Bed/2 Bath By Oxford St W1,"You'll have a great time in this beautiful, cl...","Fitzrovia is a very desirable trendy, arty and...",https://a0.muscache.com/pictures/39d5309d-fba7...,67564,...,285,78.082192,117135.0,220.0,11385,4.613685,16.47,15.258042,3.670196,137.0
3,24328,https://www.airbnb.com/rooms/24328,20250914034649,2025-09-18,previous scrape,Battersea live/work artist house,"Artist house by SW Battersea Park, bright high...","- Battersea is a quiet family area, easy acces...",https://a0.muscache.com/pictures/9194b40f-c627...,41759,...,71,19.452055,NaN,140.0,4965,4.773305,16.73,15.603012,6.088568,NaN
4,36274,https://www.airbnb.com/rooms/36274,20250914034649,2025-09-15,city scrape,Bright 1 bedroom apt off brick lane in Shoreditch,*Update June '25- Pump Installed to improve wa...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,133271,...,42,11.506849,8820.0,127.0,7469,4.633994,16.07,15.260780,0.982912,210.0


In [2]:
from pathlib import Path

Path("../data/London/warehouse").mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
conn = sqlite3.connect("../data/London/warehouse/airbnb_london.db")  # creating database

## creating dimension tables

In [4]:
# Host

dim_host = (
    listing_master[
        [
            "host_id",
            "host_name",
            "host_location",
            "host_since",
            "host_is_superhost"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

In [5]:
# creating surrogate key

dim_host.insert(
    0,
    "host_key",
    range(1, len(dim_host)+1)
)

In [6]:
dim_host.head()

,host_key,host_id,host_name,host_location,host_since,host_is_superhost
0,1,54730,Alina,"London, United Kingdom",2009-11-16,t
1,2,60302,Philippa,"Royal Borough Of Kensington And Chelsea, Unite...",2009-12-05,f
2,3,67564,Liz,"London, United Kingdom",2010-01-04,t
3,4,41759,Joe,"London, United Kingdom",2009-09-28,f
4,5,133271,Hendryks,"London, United Kingdom",2010-05-27,f


In [7]:
# property

dim_property = (
    listing_master[
        [
            "property_type",
            "room_type",
            "bedrooms",
            "bathrooms",
            "accommodates"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

In [8]:
dim_property.insert(
    0,
    "property_key",
    range(1, len(dim_property)+1)
)

In [9]:
# Neighbourhood

dim_neighbourhood = (
    listing_master[
        [
            "neighbourhood_cleansed",
            "neighbourhood_median_price",
            "neighbourhood_listing_count",
            "neighbourhood_avg_rating"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

In [10]:
dim_neighbourhood.insert(
    0,
    "neighbourhood_key",
    range(1, len(dim_neighbourhood)+1)
)

# connecting fact to dimension

In [11]:
listing_master = listing_master.merge( # host key
    dim_host[
        ["host_key", "host_id"]
    ],
    on="host_id",
    how="left"
)

In [12]:
listing_master = listing_master.merge( # property key
    dim_property,
    on=[
        "property_type",
        "room_type",
        "bedrooms",
        "bathrooms",
        "accommodates"
    ],
    how="left"
) 

In [13]:
listing_master = listing_master.merge( # neighbourhood key
    dim_neighbourhood[
        [
            "neighbourhood_key",
            "neighbourhood_cleansed"
        ]
    ],
    on="neighbourhood_cleansed",
    how="left"
)

In [14]:
fact_listing_performance = (  # creating fact table
    listing_master[
        [
            "id",

            "host_key",
            "property_key",
            "neighbourhood_key",

            "price_clean",

            "occupancy_rate",

            "estimated_revenue",

            "number_of_reviews",

            "review_scores_rating",

            "host_tenure_years",

            "price_per_bedroom"
        ]
    ]
)

In [15]:
fact_listing_performance.rename(
    columns={
        "id": "listing_id"
    },
    inplace=True
)

In [16]:
dim_host.to_sql(
    "dim_host",
    conn,
    index=False,
    if_exists="replace"
)

dim_property.to_sql(
    "dim_property",
    conn,
    index=False,
    if_exists="replace"
)

dim_neighbourhood.to_sql(
    "dim_neighbourhood",
    conn,
    index=False,
    if_exists="replace"
)

33

In [17]:
fact_listing_performance.to_sql(
    "fact_listing_performance",
    conn,
    index=False,
    if_exists="replace"
)

96871

In [18]:
pd.read_sql(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    """,
    conn
)

,name
0,dim_host
1,dim_property
2,dim_neighbourhood
3,fact_listing_performance


In [19]:
# Top revenue neighbourhoods

query = """
SELECT
    n.neighbourhood_cleansed,
    ROUND(
        AVG(f.estimated_revenue),
        2
    ) avg_revenue

FROM fact_listing_performance f

JOIN dim_neighbourhood n
ON f.neighbourhood_key = n.neighbourhood_key

GROUP BY n.neighbourhood_cleansed

ORDER BY avg_revenue DESC

LIMIT 10;
"""

pd.read_sql(query, conn)

,neighbourhood_cleansed,avg_revenue
0,Kensington And Chelsea,51383.25
1,Westminster,47114.24
2,Islington,35247.29
3,Lambeth,33373.81
4,City Of London,32336.69
5,Camden,30785.11
6,Hammersmith And Fulham,29725.84
7,Richmond Upon Thames,27917.93
8,Wandsworth,27335.85
9,Tower Hamlets,25989.76


In [20]:
# Top Hosts by revenue

query = """
SELECT
    h.host_name,

    SUM(
        f.estimated_revenue
    ) total_revenue

FROM fact_listing_performance f

JOIN dim_host h
ON f.host_key = h.host_key

GROUP BY h.host_name

ORDER BY total_revenue DESC

LIMIT 10;
"""

pd.read_sql(query, conn)

,host_name,total_revenue
0,Martin,36350737.0
1,Maxime,31295466.0
2,UniFlx,29542043.0
3,Léa,26626095.0
4,Wilson,23357081.0
5,Blueground,20895197.0
6,James,20109321.0
7,Djamshed,16945284.0
8,Alex,16345817.0
9,Pavel,14530423.0


In [21]:
# Average Price by Room Type

query = """
SELECT

    p.room_type,

    ROUND(
        AVG(f.price_clean),
        2
    ) avg_price

FROM fact_listing_performance f

JOIN dim_property p
ON f.property_key = p.property_key

GROUP BY p.room_type;
"""

pd.read_sql(query, conn)

,room_type,avg_price
0,Entire home/apt,279.35
1,Hotel room,657.83
2,Private room,121.71
3,Shared room,96.91


In [22]:
# Occupancy by neighbourhood

query = """
SELECT
    n.neighbourhood_cleansed,
    AVG(f.occupancy_rate) occupancy
FROM fact_listing_performance f
JOIN dim_neighbourhood n
ON f.neighbourhood_key = n.neighbourhood_key
GROUP BY n.neighbourhood_cleansed
ORDER BY occupancy DESC;
"""

pd.read_sql(query, conn)

,neighbourhood_cleansed,occupancy
0,Hackney,72.297360
1,Islington,69.540205
2,Lambeth,67.300868
3,Tower Hamlets,66.552673
4,Southwark,65.380389
5,Haringey,65.233320
6,Wandsworth,64.191422
7,Lewisham,63.892251
8,Richmond Upon Thames,63.627124
9,Hammersmith And Fulham,62.870876


In [23]:
# Highest Rated Neighbourhoods

query = """
SELECT
    n.neighbourhood_cleansed,
    ROUND(
        AVG(f.review_scores_rating),
        2
    ) AS avg_rating,

    COUNT(*) AS listing_count

FROM fact_listing_performance f

JOIN dim_neighbourhood n
ON f.neighbourhood_key = n.neighbourhood_key

WHERE f.review_scores_rating IS NOT NULL

GROUP BY n.neighbourhood_cleansed

HAVING COUNT(*) >= 10

ORDER BY avg_rating DESC

LIMIT 10;
"""

pd.read_sql(query, conn)

,neighbourhood_cleansed,avg_rating,listing_count
0,Richmond Upon Thames,4.81,985
1,Merton,4.79,1128
2,Kingston Upon Thames,4.78,548
3,Wandsworth,4.77,3700
4,Bromley,4.77,653
5,Waltham Forest,4.76,1422
6,Hackney,4.76,4906
7,Lambeth,4.74,4050
8,Haringey,4.74,1928
9,Bexley,4.74,429


In [24]:
# whether highly rated listings tend to earn more

query = """
SELECT
    CASE
        WHEN review_scores_rating >= 4.8 THEN 'Excellent (4.8+)'
        WHEN review_scores_rating >= 4.5 THEN 'Very Good (4.5-4.79)'
        WHEN review_scores_rating >= 4.0 THEN 'Good (4.0-4.49)'
        ELSE 'Below 4.0'
    END AS rating_category,

    COUNT(*) AS listings,

    ROUND(
        AVG(estimated_revenue),
        2
    ) AS avg_revenue

FROM fact_listing_performance

WHERE review_scores_rating IS NOT NULL

GROUP BY rating_category

ORDER BY avg_revenue DESC;
"""

pd.read_sql(query, conn)

,rating_category,listings,avg_revenue
0,Excellent (4.8+),40474,30740.30
1,Below 4.0,3115,27009.01
2,Good (4.0-4.49),10044,25764.35
3,Very Good (4.5-4.79),19116,25594.18


In [25]:
conn.close()